# 1805 context admission audit

**Joint experiment: no-go under the unchanged posture definition.** All 22 entries are retained: 44 fields from 51 source claims. Six fortification fields are eligible candidates, sixteen are blocked; all twenty-two realized-posture fields are excluded as pre-contact predictors. No model was fitted or feature admitted.

This is a companion to [the source-linked report](napoleonic-1805-context-admission-v1.md). It recounts recorded primary judgments, not an independent historical review.


## Context and methods

This is the complete frozen `third-coalition 1805` development group, selected before semantic assessment. Its first-hostile-contact boundary includes outposts; source-relative ordering may suffice without a clock.

### Key assumptions

The unchanged posture code records realized offensive action; it cannot be relabelled prior intention. Fortification is a source-qualified named-works category. `none_named` does not mean unfortified. Works beyond the first outpost hit can qualify if the engagement position was already held. The existing Civil War admission validator does not implement this proposed Napoleonic profile.


## Data

### 1. Load the frozen records

Only local pinned files are read. The three pre-assessment records and the final observations have byte bindings; no reserved-final entry or source fetch is used.


In [1]:
from pathlib import Path
import json
import hashlib
from collections import Counter
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "generalship" / "napoleonic_context_admission.py").is_file())
base = root / "data/napoleonic/context-admission-v1"
read = lambda p: json.loads(p.read_text(encoding="utf-8"))
scope, protocol, snapshot, observations = (read(base / (n + ".json")) for n in ("scope", "protocol", "snapshot", "observations"))
sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
assert observations["scope_sha256"] == sha(base / "scope.json")
assert observations["protocol_sha256"] == sha(base / "protocol.json")
assert observations["snapshot_sha256"] == sha(base / "snapshot.json")
partition = read(root / "data/napoleonic/methods-v1/evaluation.json")
ids = scope["entry_ids"]
assert ids == partition["frozen_groups"]["third-coalition 1805"]
assert not set(ids) & set(partition["reserved_final"])
print("Frozen development entries:", len(ids), "| Reserved-final entries used: 0")


Frozen development entries: 22 | Reserved-final entries used: 0


### 2. Check source and claim denominators

A citation occurrence is a pointer on an original claim. The 405 pointers contain 374 distinct citation objects. Five additional existing-source context excerpts make 379 audited passages; sources or alternative accounts never add engagements.


In [2]:
claims = observations["claims"]
keys = [(r["battle_id"], r["claim_id"]) for r in claims]
assert len(keys) == len(set(keys)) == 51
originals = {(d["battle_id"], c["id"]): c for d in snapshot["dossiers"] for c in d["claims"]}
assert set(keys) == set(originals)
citations = [c for r in originals.values() for c in r["citations"]]
unique = {json.dumps(c, ensure_ascii=False, sort_keys=True) for c in citations}
assert len(citations) == sum(len(c["citations"]) for c in claims) == 405
assert len(unique) == 374
assert len(observations["passages"]) == 379
print("Source claims:", len(claims), "| Citation occurrences:", len(citations), "| Distinct prior passages:", len(unique))
print("Additional context excerpts: 5 | Listed images re-inspected:", len(observations["inspection"]["images_reinspected"]))


Source claims: 51 | Citation occurrences: 405 | Distinct prior passages: 374
Additional context excerpts: 5 | Listed images re-inspected: 10


## Results

### 3. Recount the engagement fields

All alternatives belong to one engagement-field key. An excluded alternative does not erase another unknown. These counts summarize proposed-use judgments, not production admission decisions. Exact values are more useful than a chart for this small categorical audit.


In [3]:
fields = {}
for bid in ids:
    for dim in ("posture", "fortification"):
        rows = [c for c in claims if c["battle_id"] == bid and c["dimension"] == dim]
        assert rows
        states = {c["audit_status"] for c in rows}
        candidates = {c["original_code"] for c in rows if c["audit_status"] == "eligible_candidate"}
        status = "excluded" if states == {"excluded"} else ("blocked" if "blocked" in states or len(candidates) != 1 else "eligible_candidate")
        fields[bid, dim] = status
print("Unit                 Candidate  Blocked  Excluded  Total")
for dim in ("posture", "fortification"):
    count = Counter(s for (bid, d), s in fields.items() if d == dim)
    print(f"{dim:20} {count['eligible_candidate']:9} {count['blocked']:8} {count['excluded']:9} {sum(count.values()):6}")
print("Source-claim statuses:", dict(sorted(Counter(c["audit_status"] for c in claims).items())))
complete = [bid for bid in ids if all(fields[bid, dim] == "eligible_candidate" for dim in ("posture", "fortification"))]
assert len(fields) == 44 and complete == []
print("Complete joint entries:", len(complete), "of", len(ids))


Unit                 Candidate  Blocked  Excluded  Total
posture                      0        0        22     22
fortification                6       16         0     22
Source-claim statuses: {'blocked': 20, 'eligible_candidate': 6, 'excluded': 25}
Complete joint entries: 0 of 22


### 4. Check the material distinctions

Neresheim and Steken retain reporting-only `none_named` candidates. Caldiero and Austerlitz qualify on pre-existing held positions beyond the first contact. Mariazell has a known later occupation. Dornbirn retains a scope/location dispute; its early entrenchment construction is not mislabelled post-contact.


In [4]:
assert all(c["physical_absence"] == {"A": None, "B": None} for c in claims)
assert all(c["audit_status"] == "excluded" for c in claims if c["dimension"] == "posture")
lookup = {(c["battle_id"], c["claim_id"]): c for c in claims}
assert lookup["B368b", "fortification-schoenhals"]["reason_codes"] == ["known_postcontact_occupation"]
assert fields["B368b", "fortification"] == fields["B368d", "fortification"] == "blocked"
assert fields["B367a", "fortification"] == fields["B369c", "fortification"] == "eligible_candidate"
assert "caldiero-before-opening" in observations["passages"]
assert "parsdorf-no-blow" in observations["passages"]
assert all(observations[k] == 0 for k in ("historical_fits", "features_admitted", "model_rows_emitted", "reserved_final_entries_used"))
print("Candidate fortification entries:", ", ".join(bid for bid in ids if fields[bid, "fortification"] == "eligible_candidate"))
print("Physical-absence indicators: all null | Historical fits: 0 | Admitted features: 0")
print("Joint experiment: no-go; retain the six source-qualified fortification candidates and stop this audit.")


Candidate fortification entries: B365b, B365c, B367a, B368a, B369c, B370a
Physical-absence indicators: all null | Historical fits: 0 | Admitted features: 0
Joint experiment: no-go; retain the six source-qualified fortification candidates and stop this audit.


## Takeaways

The joint proposal fails at variable definition: actual attacks are events, not unchanged pre-contact conditions. The six fortification candidates preserve useful source-qualified descriptions; they do not establish an adequate sample or an unbiased physical-advantage predictor. All sixteen blocked fields and all source alternatives remain visible.

All 405 prior citation pointers were read; 10 of the 77 associated images were visually re-inspected here. Other image bindings preserve earlier records and are not new image checks. The full source-linked report and observations retain exact quotations, locators, source and metadata hashes, unknown dates and dependency notes. Primary verification is not historical truth or independent corroboration.

A future fortification-only or planned-role experiment would need a separate definition, coextensive force/target rows, reporting/missingness and source sensitivities, grouped evaluation, and explicit experiment authorization. No wider extraction follows from this notebook.
